# Varying Light Counts Ablation Study

This notebook provides an example of how to compare the same scene with different lighting configurations, as shown in the [demo video](https://youtu.be/px7gxgCySMQ?si=-PlnQySRZknDyHks&t=87).

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.transforms.v2 import RandomResizedCrop

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.clip import CLIPDirectionalCosineSimilarity
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion


In [ ]:
# Define light rig configurations to benchmark
configurations = ['dome_lights', 'four_small_area_lights', 'single_sun_light']
scenes_to_test = [
    CarStudioScene(configuration=config, device=device) for config in configurations
]


In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 250
global_seed = 2

clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
fine_tune_name = "siglip_blend-training-data_64-output-dim.pt"

model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
    clip_model_name,
    device=device,
    pretrained=clip_pretrained,
    fine_tune=fine_tune_name,
)
model.eval()

output_directory = "varying_light_counts_ablation"

initial_prompt = "ugly, uninteresting lighting"
target_prompt = "a bright sunny day lighting"

size = model.visual.preprocess_cfg["size"] or (224, 224) # type: ignore

for scene, configuration in zip(scenes_to_test, configurations):
    print(f"--- Running Light Count Ablation on {scene.name} ({configuration}) ---")

    criterion = CLIPDirectionalCosineSimilarity(
        initial_prompt,
        target_prompt,
        scene.get_combined_image(color_space_converter).permute(2, 1, 0),
        model,
        tokenizer,
        device=device,
        preprocess=preprocess_eval,
        always_prenormalize_vectors=True,
    )

    title_prefix = f"Light Count Ablation ({scene.name}_{configuration})"

    optimize_with_criterion(
        scene,
        lr,
        n_iter,
        criterion,
        starting_multiplier_std=(0.1, 0.1, 0.1),
        output_subdirectory_name=output_directory,
        n_results=4,
        augmentation=RandomResizedCrop(size=size, scale=(0.3, 1.0), antialias=True), # type: ignore
        render_color_space_converter=color_space_converter,
        require_physically_plausible_multipliers=True,
        title_prefix=title_prefix,
        device=device,
        save_every=50,
        model_name=clip_model_name,
        pretrained_source=fine_tune_name,
        seed=global_seed,
        show_images_after_augmentation=True,
    )
